# Python Imports

In [ ]:
import pandas as pd
import os, requests, time

from redis import Redis

# Ingest Data
- Data is stored in the `capstone 210` schema in the `canineglucose` folder.
- Set path and read into pandas.

In [ ]:
# Set path to raw datasets
datapath = '.'

In [ ]:
df = pd.read_csv('glucose_canine6.csv')

In [ ]:
# set canine ID
df['ID'] = 6

In [ ]:
# Reorganizes columns
df = df[
    ['ID', 'Device', 'Serial Number', 'Device Timestamp', 'Record Type', 'Historic Glucose mg/dL','Scan Glucose mg/dL']
]

In [ ]:
# Drop record types 5,6
df = df[df['Record Type'] != 5]
df = df[df['Record Type'] != 6]

In [ ]:
# Convert time column to datetime
df['Device Timestamp'] = pd.to_datetime(df['Device Timestamp'], format='%m/%d/%y %H:%M')

# Sort DF by ID, then by time
df = df.sort_values(['ID', 'Device Timestamp'])

In [ ]:
#Combine historic and scan glucose
df['glucose'] = df['Historic Glucose mg/dL'].fillna(df['Scan Glucose mg/dL'])

In [ ]:
# Create labels for binary classification and multi-class classification

def safe_unsafe(glucose):
  if glucose < 65 or glucose > 250:
    return 'Unsafe'
  else:
    return 'Safe'

# Multiclass
def categorize_glucose(glucose):
    if glucose < 65:
        return 'Hypoglycemia'
    elif glucose > 250:
        return 'Hyperglycemia'
    else:
        return 'Normal'

In [ ]:
df['safe_unsafe'] = df['glucose'].apply(safe_unsafe)

In [ ]:
df['glucose_category'] = df['glucose'].apply(categorize_glucose)

In [ ]:
df.drop(columns=['Historic Glucose mg/dL', 'Scan Glucose mg/dL'], inplace=True)

In [ ]:
# Drop duplicates, keeping the first measurement. This will always keep historic, and if no historic keep the first scan.
# We do this b/c there isn't that much variation in the STD of duplicated scans, as above.
df = df.drop_duplicates(['Serial Number', 'Device Timestamp'], keep='first')

In [ ]:
df

In [ ]:
df.nunique()

In [ ]:
def post_glucose_rows(
    df: pd.DataFrame,
    endpoint_url: str,
    delay_seconds: float = 0.5,
    timeout: int = 10,
) -> list[dict]:
    results = []

    for idx, row in df.iterrows():
        ts = pd.to_datetime(row["Device Timestamp"], utc=True)

        payload = {
            "Device": str(row["Device"]),
            "SerialNumber": str(row["Serial Number"]),
            "DeviceTimestamp": ts.isoformat(timespec="milliseconds").replace("+00:00", "Z"),
            "RecordType": int(row["Record Type"]),
            "Glucose": int(float(row["glucose"])),
        }

        try:
            response = requests.post(endpoint_url, json=payload, timeout=timeout)
            results.append({
                "row_index": idx,
                "status_code": response.status_code,
                "ok": response.ok,
                "response_text": response.text,
            })
        except requests.RequestException as e:
            results.append({
                "row_index": idx,
                "status_code": None,
                "ok": False,
                "response_text": str(e),
            })

        time.sleep(delay_seconds)

    return results

# Method 1: POST to URL
Simulates streaming to the platform from a CGM, except much faster than real time. No one wants to wait 15 mins for the next API call in a 5 minute demo.

- 1 POST / second (delay_seconds = 1) provides a nice snappy visualization. Just be sure the Observable dashboard is polling at 1000ms so they are synced.

In [ ]:
endpoint_url = 'url'
key = "device.id"

In [ ]:
# restults = post_glucose_rows(
#     df=df,
#     endpoint_url= endpoint_url,
#     delay_seconds = 0.001,
#     timeout = 10
# )

# Method 2: Kubectl Port Forwarding
Instead of using the API, connect directly to the database and write to it in batches.

`kubectl port-forward redis-pod-name 6379:6379`

In [ ]:
def seed_with_pipeline(redis_client, df, batch_size=1000):
    pipe = redis_client.pipeline(transaction=False)
    count = 0

    for _, row in df.iterrows():
        key = f"{row['Device']}.{row['Serial Number']}"
        ts = int(pd.to_datetime(row["Device Timestamp"], utc=True).timestamp() * 1000)
        glucose = int(float(row["glucose"]))

        pipe.execute_command("TS.ADD", key, ts, glucose)
        count += 1

        if count % batch_size == 0:
            pipe.execute()

    if count % batch_size != 0:
        pipe.execute()

In [ ]:
# redis_client = Redis("localhost", 6379, decode_responses=True)

In [ ]:
# seed_with_pipeline(redis_client, df, batch_size=5000)

# Verify Data Populated Correctly


In [ ]:
redis_client.execute_command("TS.INFO", key)

In [ ]:
# redis_client.execute_command("FLUSHALL")